In [ ]:
import os
input_dir = '/kaggle/input/human-bone-fractures-multi-modal-image-dataset'
data_yaml_path = os.path.join(input_dir, "data.yaml")

print(f"Content of {data_yaml_path}:")
with open(data_yaml_path, 'r') as f:
    print(f.read())

In [ ]:
"""
Bone Fracture Dataset Balancing, Splitting, and Augmentation Script
Configured for Kaggle Notebook Environment (Uses /kaggle/input and /kaggle/working)
"""

import os
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
import numpy as np
import yaml 
# FIX: Import image utility functions directly for use in Step 7
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array 

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# ============================================
# CONFIGURATION
# ============================================

# FIX: BALANCED_COUNTS now includes all 10 classes found in data.yaml
BALANCED_COUNTS = {
    'Comminuted': 107,
    'Greenstick': 83,
    'Healthy': 64,
    'Oblique Displaced': 107,
    'Oblique': 57,
    'Spiral': 74,
    'Transverse Displaced': 107,
    'Transverse': 107,
    'Linear': 50,    # Target count for the new class
    'Segmental': 50  # Target count for the new class
}

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX: Kaggle paths
INPUT_DIR = '/kaggle/input/human-bone-fractures-multi-modal-image-dataset'
OUTPUT_DIR = '/kaggle/working/balanced_dataset'
AUGMENTED_DIR = '/kaggle/working/balanced_augmented_dataset' 

print("=" * 60)
print("BONE FRACTURE DATASET PREPARATION FOR KAGGLE")
print("=" * 60)

# ============================================
# STEP 1 & 2: Locate and Organize Images
# ============================================

print("\n" + "=" * 60)
print("STEP 1 & 2: Locating and organizing images")
print("=" * 60)

if not os.path.exists(INPUT_DIR):
    raise FileNotFoundError(f"Dataset not found at Kaggle path: {INPUT_DIR}")

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
all_images = []
image_folder = os.path.join(INPUT_DIR, "images")

print("Scanning for images...")
if os.path.exists(image_folder):
    for file in os.listdir(image_folder):
        if Path(file).suffix.lower() in image_extensions:
            all_images.append(os.path.join(image_folder, file))
else:
    for root, dirs, files in os.walk(INPUT_DIR):
        for file in files:
            if Path(file).suffix.lower() in image_extensions:
                all_images.append(os.path.join(root, file))

if len(all_images) == 0:
    raise ValueError("No images found in dataset. Check Kaggle path.")

print(f"Found {len(all_images)} total images")

# ============================================
# STEP 3: Classify Images by Fracture Type (YOLO LOGIC - FIXED)
# ============================================

print("\n" + "=" * 60)
print("STEP 3: Classifying images by fracture type (Hardcoding fixed class mapping)")
print("=" * 60)

# FIX: Hardcode Class Mapping (Bypassing ScannerError and using all 10 classes)
class_index_to_name = {
    0: 'Comminuted',
    1: 'Greenstick',
    2: 'Healthy',
    3: 'Linear',
    4: 'Oblique Displaced',
    5: 'Oblique',
    6: 'Segmental',
    7: 'Spiral',
    8: 'Transverse Displaced',
    9: 'Transverse'
}

print(f"✓ Hardcoded {len(class_index_to_name)} class names, bypassing corrupted data.yaml.")

# --- 3b: Process Images using Label Files ---
labels_folder_path = os.path.join(INPUT_DIR, "labels")
class_images = {cls: [] for cls in BALANCED_COUNTS.keys()}
unclassified = []
total_classified = 0

print("Processing images and matching them to YOLO labels...")

for img_path in all_images:
    img_filename = os.path.basename(img_path)
    label_filename = Path(img_filename).stem + ".txt"
    label_path = os.path.join(labels_folder_path, label_filename)
    
    classified = False
    
    if os.path.exists(label_path):
        try:
            with open(label_path, 'r') as f:
                content = f.read().strip()
            
            class_name = None 
            if not content:
                # Assuming empty label file means the target is 'Healthy'
                class_name = "Healthy" 
            else:
                first_line = content.split('\n')[0].split()
                if first_line and first_line[0].isdigit():
                    class_index = int(first_line[0]) 
                    class_name = class_index_to_name.get(class_index, "Unknown")
            
            # Final check and classification
            if class_name in BALANCED_COUNTS:
                class_images[class_name].append(img_path)
                classified = True
                total_classified += 1

        except Exception as e:
            # Silently skip bad label files
            pass 
    
    if not classified:
        unclassified.append(img_path)

# Print class distribution 
print("\n📊 Detected class distribution:")
print("-" * 60)
for cls, images in class_images.items():
    count = len(images)
    target = BALANCED_COUNTS[cls]
    status = "✓" if count >= target else "⚠"
    print(f"  {status} {cls:25s}: {count:4d} images (need {target:3d})")
print("-" * 60)

missing_classes = [cls for cls, images in class_images.items() if len(images) == 0]
if missing_classes:
    print(f"\n⚠ WARNING: No images found for classes: {', '.join(missing_classes)}. Check label files.")

# ============================================
# STEP 4 & 5: Balance Dataset & Split (Train/Val/Test)
# ============================================

print("\n" + "=" * 60)
print("STEP 4 & 5: Balancing and Splitting dataset")
print("=" * 60)

balanced_images = {}
for cls, target_count in BALANCED_COUNTS.items():
    available = len(class_images.get(cls, []))
    if available < target_count:
        balanced_images[cls] = class_images.get(cls, [])
    else:
        balanced_images[cls] = random.sample(class_images[cls], target_count)
    print(f"  {cls:25s}: {len(balanced_images[cls]):4d} images selected")


splits = {'train': {}, 'val': {}, 'test': {}}
for cls, images in balanced_images.items():
    if len(images) < 3:
        splits['train'][cls] = images
        splits['val'][cls] = []
        splits['test'][cls] = []
        continue

    train_imgs, temp_imgs = train_test_split(images, train_size=TRAIN_RATIO, random_state=42)
    
    if len(temp_imgs) < 2:
        splits['train'][cls] = train_imgs
        splits['val'][cls] = temp_imgs
        splits['test'][cls] = []
    else:
        val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
        val_imgs, test_imgs = train_test_split(temp_imgs, train_size=val_ratio_adjusted, random_state=42)
        splits['train'][cls] = train_imgs
        splits['val'][cls] = val_imgs
        splits['test'][cls] = test_imgs
    
    print(f"  {cls:25s}: Train={len(splits['train'][cls]):3d}, Val={len(splits['val'][cls]):3d}, Test={len(splits['test'][cls]):3d}")

# ============================================
# STEP 6: Copy Images to New Directory Structure
# ============================================

print("\n" + "=" * 60)
print("STEP 6: Creating organized directory structure")
print("=" * 60)

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

for split in ['train', 'val', 'test']:
    for cls in BALANCED_COUNTS.keys():
        cls_dir = cls.replace(' ', '_')
        path = os.path.join(OUTPUT_DIR, split, cls_dir)
        os.makedirs(path, exist_ok=True)

total_copied = 0
for split, class_dict in splits.items():
    for cls, images in class_dict.items():
        cls_dir = cls.replace(' ', '_')
        dest_dir = os.path.join(OUTPUT_DIR, split, cls_dir)
        for i, src_path in enumerate(images):
            ext = Path(src_path).suffix
            original_stem = Path(src_path).stem 
            new_filename = f"{cls_dir}_{original_stem}_{i:04d}{ext}"
            dest_path = os.path.join(dest_dir, new_filename)
            shutil.copy2(src_path, dest_path)
            total_copied += 1

print(f"✓ Total: {total_copied} original images copied to {OUTPUT_DIR}")


# ============================================
# STEP 7: Augment Training Data
# ============================================

print("\n" + "=" * 60)
print("STEP 7: Augmenting Training Data")
print("=" * 60)

# Define Augmentation Parameters
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Setup Augmentation Directories 
if os.path.exists(AUGMENTED_DIR):
    shutil.rmtree(AUGMENTED_DIR)
os.makedirs(AUGMENTED_DIR, exist_ok=True)
AUG_TRAIN_DIR = os.path.join(AUGMENTED_DIR, 'train')
VAL_DIR = os.path.join(AUGMENTED_DIR, 'val')
TEST_DIR = os.path.join(AUGMENTED_DIR, 'test')

# Copy Val and Test sets without augmentation
shutil.copytree(os.path.join(OUTPUT_DIR, 'val'), VAL_DIR)
shutil.copytree(os.path.join(OUTPUT_DIR, 'test'), TEST_DIR)

# Augment and Save Training Data
print("Generating augmented training images...")
AUG_MULTIPLIER = 3 

total_augmented = 0
for cls in BALANCED_COUNTS.keys():
    cls_dir = cls.replace(' ', '_')
    src_dir = os.path.join(OUTPUT_DIR, 'train', cls_dir)
    dest_dir = os.path.join(AUG_TRAIN_DIR, cls_dir)
    os.makedirs(dest_dir, exist_ok=True)
    
    for filename in os.listdir(src_dir):
        img_path = os.path.join(src_dir, filename)
        
        # 3a. Copy the original image to the augmented folder
        shutil.copy2(img_path, os.path.join(dest_dir, filename))
        total_augmented += 1
        
        # 3b. Generate augmented images
        # FIX: Use standalone load_img and img_to_array functions
        img = load_img(img_path)  
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)

        i = 0
        for batch in datagen.flow(x, batch_size=1, save_to_dir=dest_dir, 
                                  save_prefix=f'aug_{Path(filename).stem}', 
                                  save_format='jpg'):
            i += 1
            total_augmented += 1
            if i >= AUG_MULTIPLIER:
                break
        
        

print(f"✓ Total images in augmented train set (Original + Augmentations): {total_augmented} ")
print(f"✅ FINAL DATASET READY AT: {AUGMENTED_DIR}")

# ============================================
# STEP 8: Create Summary Report (Optional, but useful)
# ============================================

print("\n" + "=" * 60)
print("STEP 8: Generating summary report")
print("=" * 60)

summary_data = []
for split in ['train', 'val', 'test']:
    for cls in BALANCED_COUNTS.keys():
        cls_dir = cls.replace(' ', '_')
        # Use the AUGMENTED_DIR for the final report to reflect the dataset being used for training
        path = os.path.join(AUGMENTED_DIR, split, cls_dir) 
        count = len(os.listdir(path))
        summary_data.append({
            'Split': split,
            'Class': cls,
            'Count': count
        })

df_summary = pd.DataFrame(summary_data)
summary_pivot = df_summary.pivot(index='Class', columns='Split', values='Count')

print("\n📊 Final Dataset Distribution:")
print("=" * 60)
print(summary_pivot.to_string())
print("=" * 60)

In [ ]:
"""
Bone Fracture Dataset Balancing, Splitting, and Augmentation Script
Configured for Kaggle Notebook Environment. This script is ready to run,
provided the dataset 'human-bone-fractures-multi-modal-image-dataset' is added
to your notebook via the right sidebar.
"""

import os
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
import numpy as np
import yaml 
# FIX: Import image utility functions directly for augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array 

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# ============================================
# CONFIGURATION
# ============================================

# FIX: BALANCED_COUNTS includes all 10 classes found in data.yaml
BALANCED_COUNTS = {
    'Comminuted': 107,
    'Greenstick': 83,
    'Healthy': 64,
    'Oblique Displaced': 107,
    'Oblique': 57,
    'Spiral': 74,
    'Transverse Displaced': 107,
    'Transverse': 107,
    'Linear': 50,    # Target count for the new class
    'Segmental': 50  # Target count for the new class
}

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Kaggle paths (these are correct for the Kaggle Notebook environment)
INPUT_DIR = '/kaggle/input/human-bone-fractures-multi-modal-image-dataset'
OUTPUT_DIR = '/kaggle/working/balanced_dataset'              # Original split output
AUGMENTED_DIR = '/kaggle/working/balanced_augmented_dataset' # Final dataset output

print("=" * 60)
print("BONE FRACTURE DATASET PREPARATION FOR KAGGLE")
print("=" * 60)

# ============================================
# STEP 1 & 2: Locate Images
# ============================================

if not os.path.exists(INPUT_DIR):
    raise FileNotFoundError(f"Dataset not found at Kaggle path: {INPUT_DIR}. Please add the dataset.")

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
all_images = []
image_folder = os.path.join(INPUT_DIR, "images")

if os.path.exists(image_folder):
    for file in os.listdir(image_folder):
        if Path(file).suffix.lower() in image_extensions:
            all_images.append(os.path.join(image_folder, file))

if len(all_images) == 0:
    raise ValueError("No images found in dataset. Check Kaggle path.")

print(f"✓ Found {len(all_images)} total images")

# ============================================
# STEP 3: Classify Images (Fixed YOLO Logic)
# ============================================

# FIX: Hardcode Class Mapping (Bypassing corrupted data.yaml and using all 10 classes)
class_index_to_name = {
    0: 'Comminuted', 1: 'Greenstick', 2: 'Healthy', 3: 'Linear', 
    4: 'Oblique Displaced', 5: 'Oblique', 6: 'Segmental', 7: 'Spiral', 
    8: 'Transverse Displaced', 9: 'Transverse'
}

labels_folder_path = os.path.join(INPUT_DIR, "labels")
class_images = {cls: [] for cls in BALANCED_COUNTS.keys()}

for img_path in all_images:
    img_filename = os.path.basename(img_path)
    label_filename = Path(img_filename).stem + ".txt"
    label_path = os.path.join(labels_folder_path, label_filename)
    
    if os.path.exists(label_path):
        try:
            with open(label_path, 'r') as f:
                content = f.read().strip()
            
            class_name = "Healthy" 
            if content:
                first_line = content.split('\n')[0].split()
                if first_line and first_line[0].isdigit():
                    class_index = int(first_line[0]) 
                    class_name = class_index_to_name.get(class_index)
            
            if class_name in BALANCED_COUNTS:
                class_images[class_name].append(img_path)
        except:
            pass 

# ============================================
# STEP 4 & 5: Balance Dataset & Split 
# ============================================

balanced_images = {}
for cls, target_count in BALANCED_COUNTS.items():
    available = len(class_images.get(cls, []))
    images = class_images.get(cls, [])
    if available >= target_count:
        balanced_images[cls] = random.sample(images, target_count)
    else:
        balanced_images[cls] = images

splits = {'train': {}, 'val': {}, 'test': {}}
for cls, images in balanced_images.items():
    if len(images) < 3:
        splits['train'][cls] = images
        splits['val'][cls] = []
        splits['test'][cls] = []
        continue

    # Split logic
    train_imgs, temp_imgs = train_test_split(images, train_size=TRAIN_RATIO, random_state=42)
    val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    val_imgs, test_imgs = train_test_split(temp_imgs, train_size=val_ratio_adjusted, random_state=42)
    
    splits['train'][cls] = train_imgs
    splits['val'][cls] = val_imgs
    splits['test'][cls] = test_imgs

# ============================================
# STEP 6: Copy Images to New Directory Structure
# ============================================

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create directory structure for the original split
for split in ['train', 'val', 'test']:
    for cls in BALANCED_COUNTS.keys():
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls.replace(' ', '_')), exist_ok=True)

# Copy files
for split, class_dict in splits.items():
    for cls, images in class_dict.items():
        cls_dir = cls.replace(' ', '_')
        dest_dir = os.path.join(OUTPUT_DIR, split, cls_dir)
        for i, src_path in enumerate(images):
            ext = Path(src_path).suffix
            original_stem = Path(src_path).stem 
            new_filename = f"{cls_dir}_{original_stem}_{i:04d}{ext}"
            shutil.copy2(src_path, os.path.join(dest_dir, new_filename))

# ============================================
# STEP 7: Augment Training Data & Create Final Dataset
# ============================================

# Define Augmentation Parameters
datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.1, horizontal_flip=True, brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Setup Augmentation Directories (This is your final output dataset)
if os.path.exists(AUGMENTED_DIR):
    shutil.rmtree(AUGMENTED_DIR)
os.makedirs(AUGMENTED_DIR, exist_ok=True)
AUG_TRAIN_DIR = os.path.join(AUGMENTED_DIR, 'train')

# Copy Val and Test sets to the FINAL output folder
shutil.copytree(os.path.join(OUTPUT_DIR, 'val'), os.path.join(AUGMENTED_DIR, 'val'))
shutil.copytree(os.path.join(OUTPUT_DIR, 'test'), os.path.join(AUGMENTED_DIR, 'test'))

AUG_MULTIPLIER = 3 
total_augmented = 0

# Augment and Save Training Data
for cls in BALANCED_COUNTS.keys():
    cls_dir = cls.replace(' ', '_')
    src_dir = os.path.join(OUTPUT_DIR, 'train', cls_dir)
    dest_dir = os.path.join(AUG_TRAIN_DIR, cls_dir)
    os.makedirs(dest_dir, exist_ok=True)
    
    for filename in os.listdir(src_dir):
        img_path = os.path.join(src_dir, filename)
        
        # 1. Copy the original image
        shutil.copy2(img_path, os.path.join(dest_dir, filename))
        
        # 2. Generate augmented images
        img = load_img(img_path)  
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)

        i = 0
        for batch in datagen.flow(x, batch_size=1, save_to_dir=dest_dir, 
                                  save_prefix=f'aug_{Path(filename).stem}', 
                                  save_format='jpg'):
            i += 1
            if i >= AUG_MULTIPLIER:
                break

# ============================================
# STEP 8: Final Summary and Instruction
# ============================================

# Generate Summary Data for the final output folder
summary_data = []
for split in ['train', 'val', 'test']:
    for cls in BALANCED_COUNTS.keys():
        path = os.path.join(AUGMENTED_DIR, split, cls.replace(' ', '_')) 
        count = len(os.listdir(path))
        summary_data.append({'Split': split, 'Class': cls, 'Count': count})

df_summary = pd.DataFrame(summary_data)
summary_pivot = df_summary.pivot(index='Class', columns='Split', values='Count')

print("\n" + "=" * 60)
print("✅ DATASET PREPARATION COMPLETE!")
print("=" * 60)

print(f"📊 Final Dataset Distribution (Total Images):")
print(summary_pivot.to_string())

print("\n\n💡 Next Step: Save Output as a Dataset")
print("1. Click the **'Save Version'** button (or 'Commit') at the top right.")
print("2. Wait for the notebook run to complete.")
print(f"3. Go to the **Output** tab and click **'+ New Dataset'** to save the folder: **{AUGMENTED_DIR}**")

In [ ]:
"""
Final Script for Bone Fracture Dataset Preparation on Kaggle.

1. Classifies 1539 images into 10 fracture types using YOLO labels.
2. Balances the dataset based on specified counts (using all 10 classes).
3. Splits into train/val/test.
4. Augments the training set.
"""

import os
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
import numpy as np
# Import Keras image utility functions directly
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array 

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# ============================================
# CONFIGURATION
# ============================================

# Balanced class counts, updated to include all 10 classes from data.yaml
BALANCED_COUNTS = {
    'Comminuted': 107,
    'Greenstick': 83,
    'Healthy': 64,
    'Oblique Displaced': 107,
    'Oblique': 57,
    'Spiral': 74,
    'Transverse Displaced': 107,
    'Transverse': 107,
   
}

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Kaggle paths 
INPUT_DIR = '/kaggle/input/human-bone-fractures-multi-modal-image-dataset'
OUTPUT_DIR = '/kaggle/working/balanced_dataset'              # Intermediate output
AUGMENTED_DIR = '/kaggle/working/balanced_augmented_dataset' # Final dataset output

print("=" * 60)
print("BONE FRACTURE DATASET PREPARATION FOR KAGGLE")
print("=" * 60)

# ============================================
# STEP 1: Split Original Dataset into Classes (Classification)
# ============================================

print("\n" + "=" * 60)
print("STEP 1: Locating and Classifying Images")
print("=" * 60)

if not os.path.exists(INPUT_DIR):
    raise FileNotFoundError(f"Dataset not found at: {INPUT_DIR}. Add the dataset to your notebook.")

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
all_images = []
image_folder = os.path.join(INPUT_DIR, "images")

# Locate all images
if os.path.exists(image_folder):
    for file in os.listdir(image_folder):
        if Path(file).suffix.lower() in image_extensions:
            all_images.append(os.path.join(image_folder, file))

if len(all_images) == 0:
    raise ValueError("No images found in the input dataset.")

print(f"✓ Found {len(all_images)} total images to classify.")

# Hardcode Class Mapping (Fixes the data.yaml read error)
class_index_to_name = {
    0: 'Comminuted', 1: 'Greenstick', 2: 'Healthy', 3: 'Linear', 
    4: 'Oblique Displaced', 5: 'Oblique', 6: 'Segmental', 7: 'Spiral', 
    8: 'Transverse Displaced', 9: 'Transverse'
}

labels_folder_path = os.path.join(INPUT_DIR, "labels")
class_images = {cls: [] for cls in BALANCED_COUNTS.keys()}

# Classify images using YOLO label files
for img_path in all_images:
    img_filename = os.path.basename(img_path)
    label_filename = Path(img_filename).stem + ".txt"
    label_path = os.path.join(labels_folder_path, label_filename)
    
    if os.path.exists(label_path):
        try:
            with open(label_path, 'r') as f:
                content = f.read().strip()
            
            class_name = "Healthy" 
            if content:
                first_line = content.split('\n')[0].split()
                if first_line and first_line[0].isdigit():
                    class_index = int(first_line[0]) 
                    class_name = class_index_to_name.get(class_index)
            
            if class_name in BALANCED_COUNTS:
                class_images[class_name].append(img_path)
        except:
            pass 

# Print classification output
print("\n📊 Initial Image Classification per Fracture Type:")
print("-" * 60)
total_classified = 0
for cls, images in class_images.items():
    print(f"  {cls:25s}: {len(images):4d} images")
    total_classified += len(images)
print("-" * 60)
print(f"  Total Classified: {total_classified}")

# ============================================
# STEP 2 & 3: Balance and Split Dataset
# ============================================

print("\n" + "=" * 60)
print("STEP 2 & 3: Balancing and Splitting dataset")
print("=" * 60)

balanced_images = {}
# Balance the dataset
for cls, target_count in BALANCED_COUNTS.items():
    available = len(class_images.get(cls, []))
    images = class_images.get(cls, [])
    if available >= target_count:
        # Undersample the majority class
        balanced_images[cls] = random.sample(images, target_count)
    else:
        # Use all available images for the minority class
        balanced_images[cls] = images
    
    print(f"  {cls:25s}: {len(balanced_images[cls]):4d} images selected (Target: {target_count})")

# Split the dataset
splits = {'train': {}, 'val': {}, 'test': {}}
for cls, images in balanced_images.items():
    if len(images) < 3:
        splits['train'][cls] = images
        splits['val'][cls] = []
        splits['test'][cls] = []
        continue

    # Split into Train vs (Val + Test)
    train_imgs, temp_imgs = train_test_split(images, train_size=TRAIN_RATIO, random_state=42)
    # Split (Val + Test) into Val vs Test
    val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    val_imgs, test_imgs = train_test_split(temp_imgs, train_size=val_ratio_adjusted, random_state=42)
    
    splits['train'][cls] = train_imgs
    splits['val'][cls] = val_imgs
    splits['test'][cls] = test_imgs
    
    
# Create intermediate directory structure
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

for split in ['train', 'val', 'test']:
    for cls in BALANCED_COUNTS.keys():
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls.replace(' ', '_')), exist_ok=True)

# Copy files to intermediate structure
for split, class_dict in splits.items():
    for cls, images in class_dict.items():
        cls_dir = cls.replace(' ', '_')
        dest_dir = os.path.join(OUTPUT_DIR, split, cls_dir)
        for i, src_path in enumerate(images):
            ext = Path(src_path).suffix
            original_stem = Path(src_path).stem 
            new_filename = f"{cls_dir}_{original_stem}_{i:04d}{ext}"
            shutil.copy2(src_path, os.path.join(dest_dir, new_filename))

# ============================================
# STEP 4: Augment the Dataset (Training Set Only)
# ============================================

print("\n" + "=" * 60)
print("STEP 4: Augmenting Training Data")
print("=" * 60)

# Define Augmentation Parameters
datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.1, horizontal_flip=True, brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Setup FINAL Output Directories
if os.path.exists(AUGMENTED_DIR):
    shutil.rmtree(AUGMENTED_DIR)
os.makedirs(AUGMENTED_DIR, exist_ok=True)
AUG_TRAIN_DIR = os.path.join(AUGMENTED_DIR, 'train')

# Copy Val and Test sets to the FINAL output folder
shutil.copytree(os.path.join(OUTPUT_DIR, 'val'), os.path.join(AUGMENTED_DIR, 'val'))
shutil.copytree(os.path.join(OUTPUT_DIR, 'test'), os.path.join(AUGMENTED_DIR, 'test'))

AUG_MULTIPLIER = 3 # Generate 3 augmented images per original training image
total_augmented_images = 0

# Augment and Save Training Data
for cls in BALANCED_COUNTS.keys():
    cls_dir = cls.replace(' ', '_')
    src_dir = os.path.join(OUTPUT_DIR, 'train', cls_dir)
    dest_dir = os.path.join(AUG_TRAIN_DIR, cls_dir)
    os.makedirs(dest_dir, exist_ok=True)
    
    for filename in os.listdir(src_dir):
        img_path = os.path.join(src_dir, filename)
        
        # 1. Copy the original image
        shutil.copy2(img_path, os.path.join(dest_dir, filename))
        total_augmented_images += 1
        
        # 2. Generate augmented images (3 copies)
        img = load_img(img_path)  
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)

        i = 0
        for batch in datagen.flow(x, batch_size=1, save_to_dir=dest_dir, 
                                  save_prefix=f'aug_{Path(filename).stem}', 
                                  save_format='jpg'):
            i += 1
            total_augmented_images += 1
            if i >= AUG_MULTIPLIER:
                break

# ============================================
# FINAL OUTPUT REPORT
# ============================================

# Generate Summary Data
summary_data = []
for split in ['train', 'val', 'test']:
    for cls in BALANCED_COUNTS.keys():
        path = os.path.join(AUGMENTED_DIR, split, cls.replace(' ', '_')) 
        count = len(os.listdir(path))
        summary_data.append({'Split': split, 'Class': cls, 'Count': count})

df_summary = pd.DataFrame(summary_data)
summary_pivot = df_summary.pivot(index='Class', columns='Split', values='Count')

print("\n" + "=" * 60)
print("✅ DATASET PREPARATION COMPLETE!")
print(f"The final dataset is located at: {AUGMENTED_DIR}")
print("=" * 60)

print(f"📊 Final Dataset Distribution (Original + Augmentations):")
print(summary_pivot.to_string())

print("\n**Next Step: Create your permanent output dataset:**")
print("1. Click the **'Save Version'** button (or **'Commit'**) at the top right.")
print("2. Wait for the notebook run to complete successfully.")
print("3. Go to the **Output** tab of the finished run and click **'+ New Dataset'**.")

In [ ]:
import os
import pandas as pd

# This path MUST match the final output directory from your previous script
AUGMENTED_DIR = '/kaggle/working/balanced_augmented_dataset'
CLASSES = [
    'Comminuted', 'Greenstick', 'Healthy', 'Oblique Displaced', 'Oblique', 
    'Spiral', 'Transverse Displaced', 'Transverse', 'Linear', 'Segmental'
]

summary_data = []
total_files = 0

print("=" * 60)
print(f"Inspecting contents of: {AUGMENTED_DIR}")
print("=" * 60)

# Walk through the directory structure: /balanced_augmented_dataset/split/class_name
for split in ['train', 'val', 'test']:
    split_path = os.path.join(AUGMENTED_DIR, split)
    
    # Check if the split directory exists
    if not os.path.exists(split_path):
        print(f"❌ WARNING: Split directory not found: {split_path}")
        continue
        
    for cls in CLASSES:
        cls_dir = cls.replace(' ', '_')
        path = os.path.join(split_path, cls_dir) 
        
        count = 0
        if os.path.exists(path):
            count = len(os.listdir(path))
            total_files += count

        summary_data.append({
            'Split': split,
            'Class': cls,
            'Count': count
        })
        
        # Print a quick check for non-empty folders
        if count > 0:
             print(f"Found {count:5d} files in -> {split}/{cls_dir}")

# ============================================
# FINAL SUMMARY TABLE
# ============================================

df_summary = pd.DataFrame(summary_data)
summary_pivot = df_summary.pivot(index='Class', columns='Split', values='Count')

print("\n" + "=" * 60)
print("📊 FINAL AUGMENTED DATASET DISTRIBUTION")
print(f"Total files processed: {total_files}")
print("=" * 60)
print(summary_pivot.to_string())
print("=" * 60)

In [ ]:
# Force uninstall and reinstall to ensure compatibility
!pip install --upgrade transformers huggingface-hub accelerate -q

In [ ]:
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Dense, Dropout, Input, GlobalAveragePooling2D, Lambda
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from transformers import TFAutoModelForImageClassification, AutoImageProcessor
# import os
# import pandas as pd
# import numpy as np

# # ====================================================
# # 1. FINAL TF CONFIGURATION AND ENVIRONMENT SETUP
# # ====================================================

# # Enable soft device placement for robustness, but the core fix is the device scope.
# print("Setting TensorFlow soft device placement...")
# tf.config.set_soft_device_placement(True)

# # Configuration and Paths (Unchanged)
# MODEL_ID = "facebook/convnextv2-huge-22k-384"
# AUGMENTED_DIR = '/kaggle/working/balanced_augmented_dataset'
# IMG_SIZE = (384, 384) 
# BATCH_SIZE = 16       
# EPOCHS = 10           
# TRAIN_DIR = os.path.join(AUGMENTED_DIR, 'train')
# VAL_DIR = os.path.join(AUGMENTED_DIR, 'val')
# TEST_DIR = os.path.join(AUGMENTED_DIR, 'test')

# print("Configuration complete. Preparing data generators...")

# # ====================================================
# # 2. DATA GENERATORS (Unchanged)
# # ====================================================

# raw_datagen = ImageDataGenerator() 

# train_generator = raw_datagen.flow_from_directory(
#     TRAIN_DIR,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='categorical',
#     shuffle=True
# )

# val_generator = raw_datagen.flow_from_directory(
#     VAL_DIR,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='categorical',
#     shuffle=False
# )

# test_generator = raw_datagen.flow_from_directory(
#     TEST_DIR,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='categorical',
#     shuffle=False
# )

# class_names = list(train_generator.class_indices.keys())
# NUM_CLASSES = len(class_names) 

# id2label = {v: k for k, v in train_generator.class_indices.items()}
# label2id = {k: v for k, v in train_generator.class_indices.items()}

# print(f"Loaded {NUM_CLASSES} classes: {class_names}")

# # ====================================================
# # 3. LOAD MODEL (FORCING GPU PLACEMENT)
# # ====================================================

# print(f"\nLoading Hugging Face model: {MODEL_ID}")

# # Determine target device for loading
# target_device = "/cpu:0"
# if tf.config.list_physical_devices('GPU'):
#     target_device = "/gpu:0"
#     print("✅ Forcing model load onto GPU:0 to prevent cross-device errors.")
# else:
#     print("⚠️ GPU not detected. Loading onto CPU.")


# # 🛑 FINAL FIX: Load the model directly onto the required device.
# with tf.device(target_device):
#     hf_model_full = TFAutoModelForImageClassification.from_pretrained(
#     	MODEL_ID,
#     	from_pt=True,
#         ignore_mismatched_sizes=True 
#     )

# # Extract the backbone layer object
# hf_model_backbone = hf_model_full.layers[0] 
# hf_model_backbone.trainable = False


# # ====================================================
# # 4. BUILD, COMPILE, AND TRAIN (Functional API)
# # ====================================================

# print("\nBuilding and compiling Keras model...")

# # Keras Functional API definition
# input_tensor = Input(shape=IMG_SIZE + (3,), name='image_input')

# # 1. Custom Preprocessing Layer
# x = Lambda(lambda img: tf.image.convert_image_dtype(img, tf.float32) * 255.0, name='preprocess')(input_tensor)

# # Output dimensions
# OUTPUT_SPATIAL_SIZE = 12
# OUTPUT_CHANNELS = 2816 
# OUTPUT_SHAPE = (OUTPUT_SPATIAL_SIZE, OUTPUT_SPATIAL_SIZE, OUTPUT_CHANNELS)


# def call_hf_backbone(inputs):
#     # Transpose input: (B, H, W, C) -> (B, C, H, W)
#     transposed_inputs = tf.transpose(inputs, perm=[0, 3, 1, 2])
    
#     # Get features from backbone (output is (B, C, H, W))
#     features = hf_model_backbone(transposed_inputs, training=False)[0] 
    
#     # Transpose output back: (B, C, H, W) -> (B, H, W, C)
#     return tf.transpose(features, perm=[0, 2, 3, 1])


# x = Lambda(
#     call_hf_backbone, 
#     output_shape=OUTPUT_SHAPE,
#     name='hf_backbone_wrapper'
# )(x) 

# # 3. Custom Classification Head
# x = GlobalAveragePooling2D(name='pooling')(x) 
# x = Dense(512, activation='relu', name='dense_1')(x)
# x = Dropout(0.3, name='dropout_1')(x)
# output_tensor = Dense(NUM_CLASSES, activation='softmax', name='final_classifier')(x)

# # Create the final Model object and compile
# model = Model(inputs=input_tensor, outputs=output_tensor)
    
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=5e-5), 
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()

# # Define callbacks
# callbacks = [
#     EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
#     ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6),
#     ModelCheckpoint('best_convnextv2_hf_fracture_classifier.h5', monitor='val_loss', save_best_only=True)
# ]

# print("\nStarting training...")

# # Start training.
# history = model.fit(
#     train_generator,
#     epochs=EPOCHS,
#     validation_data=val_generator,
#     callbacks=callbacks,
#     verbose=1
# )

# # ====================================================
# # 5. EVALUATION
# # ====================================================

# print("\n" + "=" * 60)
# print("FINAL MODEL EVALUATION ON TEST SET")
# print("=" * 60)

# model.load_weights('best_convnextv2_hf_fracture_classifier.h5')

# loss, accuracy = model.evaluate(test_generator, verbose=0)

# print(f"Test Loss: {loss:.4f}")
# print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Dense, Dropout, Input, GlobalAveragePooling2D, Lambda
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from transformers import TFAutoModelForImageClassification, AutoImageProcessor
# import os

# # ====================================================
# # 1. TF CONFIGURATION AND ENVIRONMENT SETUP
# # ====================================================

# # 🛑 CORE FIX: Hide the GPU from TensorFlow so it defaults to CPU for ALL operations.
# # This prevents the cross-device access error. Training will be slow on the CPU.
# tf.config.set_visible_devices([], 'GPU') 
# print("Running exclusively on CPU. GPU is hidden.")
# tf.config.set_soft_device_placement(True)

# # ⭐️ CONFIGURATION CHANGE FOR EVA-02
# MODEL_ID = "BAAI/EVA-02-L-Inpaint-multiscale" 
# IMG_SIZE = (336, 336) # Common size for this EVA-02 model

# # Paths (Unchanged)
# AUGMENTED_DIR = '/kaggle/working/balanced_augmented_dataset'
# BATCH_SIZE = 16       
# EPOCHS = 10           
# TRAIN_DIR = os.path.join(AUGMENTED_DIR, 'train')
# VAL_DIR = os.path.join(AUGMENTED_DIR, 'val')
# TEST_DIR = os.path.join(AUGMENTED_DIR, 'test')

# print("Configuration complete. Preparing data generators...")

# # ====================================================
# # 2. DATA GENERATORS (Unchanged)
# # ====================================================

# raw_datagen = ImageDataGenerator() 

# # WARNING: This will fail if the directory does not exist.
# # Make sure your data is available at AUGMENTED_DIR.
# train_generator = raw_datagen.flow_from_directory(
#     TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
# )
# val_generator = raw_datagen.flow_from_directory(
#     VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
# )
# test_generator = raw_datagen.flow_from_directory(
#     TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
# )

# class_names = list(train_generator.class_indices.keys())
# NUM_CLASSES = len(class_names) 
# print(f"Loaded {NUM_CLASSES} classes: {class_names}")

# # ====================================================
# # 3. LOAD MODEL (CPU Only)
# # ====================================================

# print(f"\nLoading Hugging Face EVA-02 model on CPU: {MODEL_ID}")

# # Load model directly—it will default to CPU since GPU is hidden.
# hf_model_full = TFAutoModelForImageClassification.from_pretrained(
#     MODEL_ID, from_pt=True, ignore_mismatched_sizes=True 
# )

# # Extract the backbone layer object
# hf_model_backbone = hf_model_full.layers[0] 
# hf_model_backbone.trainable = False


# # ====================================================
# # 4. BUILD, COMPILE, AND TRAIN (Functional API on CPU)
# # ====================================================

# print("\nBuilding and compiling Keras model...")

# # Keras Functional API definition
# input_tensor = Input(shape=IMG_SIZE + (3,), name='image_input')

# # 1. Custom Preprocessing Layer
# x = Lambda(lambda img: tf.image.convert_image_dtype(img, tf.float32) * 255.0, name='preprocess')(input_tensor)


# def call_hf_backbone(inputs):
#     # 1. Transpose input: (B, H, W, C) -> (B, C, H, W) for ViT/HF
#     transposed_inputs = tf.transpose(inputs, perm=[0, 3, 1, 2])
    
#     # 2. Get features from backbone. ViT models typically output a sequence of tokens 
#     # (B, Sequence_Length, Hidden_Size). We usually take the last hidden state (index 0).
#     # We pass the input and get the hidden states.
#     features = hf_model_backbone(transposed_inputs, training=False).last_hidden_state
    
#     # We take the first token (the CLS token) which is often used for classification, 
#     # instead of doing GlobalAveragePooling over a 3D tensor.
#     # Shape is now (B, Hidden_Size)
#     return features[:, 0, :] 


# # ⭐️ IMPORTANT: Output shape is now just the feature vector size (e.g., 1024)
# # We use a Lambda wrapper without a fixed output_shape for maximum compatibility, 
# # relying on Keras's shape inference for the final 1D vector.
# x = Lambda(
#     call_hf_backbone, 
#     # Removed output_shape=... - Keras can often infer the 1D vector shape (None, Hidden_Size)
#     name='hf_backbone_wrapper'
# )(x) 

# # 3. Custom Classification Head
# # Note: GlobalAveragePooling2D is removed because the Lambda now outputs a 1D vector.
# # The CLS token (x) already has the shape (None, Hidden_Size).
# x = Dense(512, activation='relu', name='dense_1')(x)
# x = Dropout(0.3, name='dropout_1')(x)
# output_tensor = Dense(NUM_CLASSES, activation='softmax', name='final_classifier')(x)

# # Create the final Model object and compile
# model = Model(inputs=input_tensor, outputs=output_tensor)
    
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=5e-5), 
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()

# # Define callbacks
# callbacks = [
#     EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
#     ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6),
#     ModelCheckpoint('best_eva02_hf_classifier.h5', monitor='val_loss', save_best_only=True)
# ]

# print("\nStarting training...")

# # Start training. (Will run on CPU)
# history = model.fit(
#     train_generator,
#     epochs=EPOCHS,
#     validation_data=val_generator,
#     callbacks=callbacks,
#     verbose=1
# )

# # ====================================================
# # 5. EVALUATION
# # ====================================================

# print("\n" + "=" * 60)
# print("FINAL MODEL EVALUATION ON TEST SET")
# print("=" * 60)

# model.load_weights('best_eva02_hf_classifier.h5')

# loss, accuracy = model.evaluate(test_generator, verbose=0)

# print(f"Test Loss: {loss:.4f}")
# print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
from huggingface_hub import login
# This will open a prompt where you must paste the token you copied from your Hugging Face settings.
login()

In [ ]:
import tensorflow as tf
from tensorflow import keras
# Import all required Keras components directly for clean usage
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, Lambda
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Import callbacks directly to resolve the NameError
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from transformers import TFAutoModelForImageClassification
import os

# ====================================================
# 1. TF CONFIGURATION AND ENVIRONMENT SETUP
# ====================================================

# 🛑 CORE FIX: Hide the GPU to enforce CPU execution and prevent device placement errors.
tf.config.set_visible_devices([], 'GPU') 
print("Running exclusively on CPU. GPU is hidden.")
tf.config.set_soft_device_placement(True)

# ⭐️ RELIABLE MODEL CONFIGURATION: ViT-Base 224
MODEL_ID = "google/vit-base-patch16-224" 
IMG_SIZE = (224, 224)              

# --- Define Your Dataset Variables Here (Assumed correct) ---
AUGMENTED_DIR = '/kaggle/working/balanced_augmented_dataset'
BATCH_SIZE = 16       
EPOCHS = 10           

TRAIN_DIR = os.path.join(AUGMENTED_DIR, 'train')
VAL_DIR = os.path.join(AUGMENTED_DIR, 'val')
TEST_DIR = os.path.join(AUGMENTED_DIR, 'test')


# ====================================================
# 2. DATA GENERATORS (REQUIRED FOR RUNNING)
# ====================================================

raw_datagen = ImageDataGenerator(rescale=1./255.) 

try:
    # NOTE: target_size MUST match IMG_SIZE (224, 224)
    train_generator = raw_datagen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
    )
    val_generator = raw_datagen.flow_from_directory(
        VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
    )
    test_generator = raw_datagen.flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
    )

    NUM_CLASSES = train_generator.num_classes
    print(f"Data generators loaded. Detected {NUM_CLASSES} classes.")
    
except Exception as e:
    print("\n--- FATAL DATA ERROR ---")
    print("Could not load data generators. Check your data paths.")
    raise FileNotFoundError("Check dataset paths and ensure data exists.")


# ====================================================
# 3. LOAD ViT MODEL (NATIVE TF)
# ====================================================

print(f"\nLoading native TF ViT model on CPU: {MODEL_ID} (Retrying after import fix)")

# This model is reliably available in TF format.
hf_model_full = TFAutoModelForImageClassification.from_pretrained(
    MODEL_ID, 
    from_pt=True, # Keeping from_pt=True to resolve the previous SafeTensors bug
    ignore_mismatched_sizes=True,
    trust_remote_code=True
)

hf_model_backbone = hf_model_full.layers[0] 
hf_model_backbone.trainable = False


# ====================================================
# 4. BUILD, COMPILE, AND TRAIN (Functional API)
# ====================================================

print("\nBuilding and compiling Keras model...")

# Input is imported directly and can be used without prefix
input_tensor = Input(shape=IMG_SIZE + (3,), name='image_input')

# 1. Custom Preprocessing Layer: Scale data to 0-255
x = Lambda(lambda img: tf.image.convert_image_dtype(img * 255.0, tf.float32), name='preprocess')(input_tensor)


def call_hf_backbone(inputs):
    # 1. Transpose input: (B, H, W, C) -> (B, C, H, W) 
    transposed_inputs = tf.transpose(inputs, perm=[0, 3, 1, 2])
    
    # 2. Get features (last hidden state)
    features = hf_model_backbone(transposed_inputs, training=False).last_hidden_state
    
    # 3. ViT pooling: Use the CLS token (first token)
    return features[:, 0, :] 


# 2. Integrate the ViT backbone via Lambda
x = Lambda(call_hf_backbone, name='hf_backbone_wrapper')(x) 

# 3. Custom Classification Head
x = Dense(512, activation='relu', name='dense_1')(x)
x = Dropout(0.3, name='dropout_1')(x)
output_tensor = Dense(NUM_CLASSES, activation='softmax', name='final_classifier')(x)

# Create the final Model object (imported directly)
model = Model(inputs=input_tensor, outputs=output_tensor)
    
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Define callbacks (all imported directly and defined here)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True), # FIXED NameError
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6), # FIXED NameError
    ModelCheckpoint('best_vit_hf_classifier_224.h5', monitor='val_loss', save_best_only=True) # FIXED NameError
]

print("\nStarting training...")

# Start training.
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

# ====================================================
# 5. EVALUATION
# ====================================================

print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION ON TEST SET")
print("=" * 60)

model.load_weights('best_vit_hf_classifier_224.h5')

loss, accuracy = model.evaluate(test_generator, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")